In [44]:
import pandas as pd
import os
import json
import copy
import re
import torch
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from pathlib import Path
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv

In [45]:
load_dotenv()

DATA_PATH = Path("./data")
RAW_1P_DATA_PATH = DATA_PATH / "input" / "aei_raw_1p_api_2026-02-05_to_2026-02-12.csv"
RAW_CLAUDE_DATA_PATH = DATA_PATH / "input" / "aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
PROMPTS_DIR = Path("./prompts")
WORK_RELATED_PROMPT_PATH = PROMPTS_DIR / "work_related.json"

OUTPUT_DIR = DATA_PATH / "output"
WORK_RELATED_OUTPUT_PATH = OUTPUT_DIR / "work_related.csv"
HIERARCHY_OUTPUT_PATH = OUTPUT_DIR / "hierarchy.csv"

WILDCHAT_DATASET = "allenai/WildChat-4.8M"
CONVERSATION_FIELD = "conversation"
LANGUAGE_FIELD = "language"
TARGET_LANGUAGE = "English"

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

RANDOM_SEED = 42
BUFFER_SIZE = 10_000
SAMPLE_SIZE = 10
MODEL_ID = "nvidia/nemotron-3-super-120b-a12b:free"
EMBEDDING_MODEL_ID = "sentence-transformers/all-mpnet-base-v2"
EMBEDDING_BATCH_SIZE = 256

PARENT_COLUMN = "parent_id"
COUNT_VARIABLE = "onet_task_count"

In [46]:
device = ""

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [47]:
wildchat_ds = load_dataset(
    WILDCHAT_DATASET,
    split="train",
    streaming=True
)
english_conversations_rows = wildchat_ds.filter(lambda x: x[LANGUAGE_FIELD] == TARGET_LANGUAGE)
sample_conversations_rows = pd.DataFrame(english_conversations_rows.shuffle(
    seed=RANDOM_SEED,
    buffer_size=BUFFER_SIZE
).take(SAMPLE_SIZE))
sample_conversations = sample_conversations_rows[[CONVERSATION_FIELD]]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

In [48]:
sample_df = pd.DataFrame(sample_conversations.head())
sample_df.head()

,conversation
0,[{'content': 'i own a rusty 1975 plymouth fury...
1,[{'content': 'Make the first part of the Delta...
2,[{'content': 'How to be a good police officer ...
3,[{'content': 'freedom planet all characters re...
4,[{'content': 'what to positively answer to fol...


In [49]:
client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

In [50]:
def get_gpt_response(messages: list[dict[str, str]]) -> str:
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
    )
    dirty_result = response.choices[0].message.content.strip()
    cleaned_result = re.search(r'<answer>(.*?)</answer>', dirty_result, re.DOTALL)
    if cleaned_result:
        return cleaned_result.group(1)
    return ""


In [51]:
def get_messages(path: Path) -> list[dict[str, str]]:
    with open(path, "r") as f:
        return json.load(f)

In [52]:
def format_conversation(conversation: list[dict[str, str]]) -> str:
    formatted = ""
    for message in conversation:
        role = message["role"]
        content = message["content"]
        formatted += f"{role}: {content}\n"
    return formatted

In [53]:
def build_messages(template: list[dict[str, str]], **kwargs):
    msgs = copy.deepcopy(template)
    for msg in msgs:
        for key, value in kwargs.items():
            if "{" + key + "}" in msg["content"]:
                msg["content"] = msg["content"].replace("{" + key + "}", value)
    return msgs

In [54]:
def filter_work_conversations(conversations: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame()
    template = get_messages(WORK_RELATED_PROMPT_PATH)
    for index, row in conversations.iterrows():
        raw_conversation = row["conversation"]
        conversation = format_conversation(conversation=raw_conversation)
        messages = build_messages(template=template, conversation=conversation)
        response = get_gpt_response(messages=messages)
        record = [{"conversation": conversation, "is_work_related": response}]
        result = pd.concat([result, pd.DataFrame(record)], ignore_index=True)
    return result

In [55]:
work_related_conversations = filter_work_conversations(conversations=sample_conversations)
work_related_conversations.to_csv(WORK_RELATED_OUTPUT_PATH)
work_related_conversations.head()

,conversation,is_work_related
0,user: i own a rusty 1975 plymouth fury beater....,No
1,user: Make the first part of the Delta Systems...,Yes
2,user: How to be a good police officer \nassist...,Yes
3,user: freedom planet all characters react to g...,No
4,user: what to positively answer to following j...,Yes


In [56]:
raw_1p_api_df = pd.read_csv(RAW_1P_DATA_PATH)
raw_claude_api_df = pd.read_csv(RAW_CLAUDE_DATA_PATH)

In [57]:
raw_claude_api_df.head()

,geo_id,geography,date_start,date_end,platform_and_product,facet,level,variable,cluster_name,value
0,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,directive,15.000000
1,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_pct,directive,20.270270
2,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,learning,20.000000
3,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_pct,learning,27.027027
4,AD,country,2026-02-05,2026-02-12,"Claude AI (Free, Pro, and Max)",collaboration,0,collaboration_count,not_classified,16.000000


In [58]:
columns_to_keep = ["level", "cluster_name"]
clusters_df = (
    raw_claude_api_df[columns_to_keep]
   .drop_duplicates()
   .reset_index(drop=True)
   .iloc[50:]
)

clusters_df.head()

,level,cluster_name
50,0,"present investment information, such as produc..."
51,0,"provide advice to clients on a contract basis,..."
52,0,"provide clients with communication assistance,..."
53,0,provide feedback and recommendations to develo...
54,0,provide information and advice to the public r...


In [59]:
model = SentenceTransformer(EMBEDDING_MODEL_ID).to(device=device)

# TODO: Store the embeddings, so that we don't have to recompute them every time.
embeddings = model.encode(
    clusters_df["cluster_name"].to_list(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)
clusters_df["embedding"] = list(embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

In [60]:
level_0 = clusters_df[clusters_df["level"] == 0]
level_1 = clusters_df[clusters_df["level"] == 1]
level_2 = clusters_df[clusters_df["level"] == 2]

level_2[PARENT_COLUMN] = None

/var/folders/tc/wc9p1tw950bb9449rjnj5n680000gn/T/ipykernel_1633/1931026715.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  level_2[PARENT_COLUMN] = None


In [61]:
level_2.head()

,level,cluster_name,embedding,parent_id
210,2,"Assist with SQL databases, data analysis, and ...","[-0.03694914, 0.019519594, -0.059328992, -0.01...",None
211,2,"Assist with academic research, scientific writ...","[0.051645294, -0.03381015, -0.007419883, -0.04...",None
212,2,"Assist with business development, analysis, re...","[0.031849258, 0.0071852207, -0.042096466, -0.0...",None
213,2,"Assist with civic information, government serv...","[-0.0074762586, 0.091707915, -0.01566061, 0.00...",None
214,2,"Assist with creative content, marketing, and m...","[0.046204854, 0.011296493, -0.030031018, -0.07...",None


In [66]:
def assign_parents(parents: pd.DataFrame, children: pd.DataFrame):
    children_matrix = np.stack(children["embedding"])
    parent_matrix = np.stack(parents["embedding"])
    similarity_matrix = cosine_similarity(children_matrix, parent_matrix)
    most_similar_parent_indices = similarity_matrix.argmax(axis=1)
    children[PARENT_COLUMN] = parents.index[most_similar_parent_indices]
    return children

In [67]:
level_0 = assign_parents(parents=level_1, children=level_0)
level_1 = assign_parents(parents=level_2, children=level_1)
hierarchy_df = pd.concat([level_0, level_1, level_2], ignore_index=True)
hierarchy_df.to_csv(HIERARCHY_OUTPUT_PATH, index=False)
hierarchy_df.head()

/var/folders/tc/wc9p1tw950bb9449rjnj5n680000gn/T/ipykernel_1633/1814519891.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  children[PARENT_COLUMN] = parents.index[most_similar_parent_indices]
/var/folders/tc/wc9p1tw950bb9449rjnj5n680000gn/T/ipykernel_1633/1814519891.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  children[PARENT_COLUMN] = parents.index[most_similar_parent_indices]


,level,cluster_name,embedding,parent_id
0,0,"present investment information, such as produc...","[-0.029537324, -0.044756297, -0.0112805795, -0...",31842
1,0,"provide advice to clients on a contract basis,...","[0.06747445, -0.010477222, 0.0012193361, -0.06...",33439
2,0,"provide clients with communication assistance,...","[0.049443517, 0.0029836611, -0.00951981, 0.001...",33316
3,0,provide feedback and recommendations to develo...,"[0.06489916, 0.032675363, -0.0359315, -0.04906...",31611
4,0,provide information and advice to the public r...,"[0.07341507, 0.014599926, -0.019335385, -0.014...",33418
